In [1]:
# семинар: скорость и память в numpy/scipy против чистого python
# цель: наглядно показать преимущества векторизации, непрерывной памяти и готовых алгоритмов

import sys
import time
import timeit
import math
import numpy as np
from scipy import linalg, optimize

# для красивого вывода таблиц (необязательно, но удобно)
try:
    import pandas as pd
    use_pandas = True
except ImportError:
    use_pandas = False

# ------------------------------------------------------------
# утилиты для замеров
# ------------------------------------------------------------

def measure_time(func, *args, repeats=5):
    """возвращает среднее время выполнения func(*args) в секундах"""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        func(*args)
        end = time.perf_counter()
        times.append(end - start)
    return min(times)  # берём лучшее время

def measure_cpu_time(func, *args):
    """процессорное время (user+system)"""
    start = time.process_time()
    func(*args)
    end = time.process_time()
    return end - start

# ------------------------------------------------------------
# 1. поэлементные операции: списки vs массивы
# ------------------------------------------------------------

print("="*60)
print("1. поэлементное сложение двух векторов")
print("="*60)

size = 10_000_000
list_a = list(range(size))
list_b = list(range(size))
arr_a = np.arange(size, dtype=np.int64)
arr_b = np.arange(size, dtype=np.int64)

def add_lists():
    return [list_a[i] + list_b[i] for i in range(size)]

def add_arrays():
    return arr_a + arr_b

t_list = measure_time(add_lists)
t_arr = measure_time(add_arrays)
print(f"списки:      {t_list:.3f} сек")
print(f"numpy:        {t_arr:.3f} сек")
print(f"ускорение:    {t_list/t_arr:.1f}x")

# ------------------------------------------------------------
# 2. скалярное произведение (dot product)
# ------------------------------------------------------------

print("\n" + "="*60)
print("2. скалярное произведение векторов")
print("="*60)

size_dot = 5_000_000
list_u = [float(i) for i in range(size_dot)]
list_v = [float(i) for i in range(size_dot)]
arr_u = np.arange(size_dot, dtype=np.float64)
arr_v = np.arange(size_dot, dtype=np.float64)

def dot_lists():
    s = 0.0
    for i in range(len(list_u)):
        s += list_u[i] * list_v[i]
    return s

def dot_arrays():
    return np.dot(arr_u, arr_v)

t_dot_list = measure_time(dot_lists)
t_dot_arr = measure_time(dot_arrays)
print(f"списки:      {t_dot_list:.3f} сек")
print(f"numpy:        {t_dot_arr:.3f} сек")
print(f"ускорение:    {t_dot_list/t_dot_arr:.1f}x")

# ------------------------------------------------------------
# 3. умножение матриц
# ------------------------------------------------------------

print("\n" + "="*60)
print("3. умножение матриц (1000x1000)")
print("="*60)

n_mat = 1000
A_list = [[float(i+j) for j in range(n_mat)] for i in range(n_mat)]
B_list = [[float(i-j) for j in range(n_mat)] for i in range(n_mat)]
A_np = np.array(A_list, dtype=np.float64)
B_np = np.array(B_list, dtype=np.float64)

def matmul_lists():
    n = len(A_list)
    C = [[0.0]*n for _ in range(n)]
    for i in range(n):
        for k in range(n):
            aik = A_list[i][k]
            rowB = B_list[k]
            rowC = C[i]
            for j in range(n):
                rowC[j] += aik * rowB[j]
    return C

# оптимизированный порядок (ikj) для списков, чтобы показать лучший вариант
def matmul_lists_ikj():
    n = len(A_list)
    C = [[0.0]*n for _ in range(n)]
    for i in range(n):
        Ci = C[i]
        Ai = A_list[i]
        for k in range(n):
            aik = Ai[k]
            Bk = B_list[k]
            for j in range(n):
                Ci[j] += aik * Bk[j]
    return C

def matmul_arrays():
    return A_np @ B_np

print("измеряем время (минуту терпения)...")
t_mat_list = measure_time(matmul_lists_ikj, repeats=3)
t_mat_arr = measure_time(matmul_arrays, repeats=3)
print(f"списки (ikj): {t_mat_list:.2f} сек")
print(f"numpy (BLAS):  {t_mat_arr:.2f} сек")
print(f"ускорение:     {t_mat_list/t_mat_arr:.1f}x")

# ------------------------------------------------------------
# 4. broadcasting: вычитание среднего из строк матрицы
# ------------------------------------------------------------

print("\n" + "="*60)
print("4. центрирование матрицы (вычитание среднего по каждой строке)")
print("="*60)

rows, cols = 5000, 100
matrix = np.random.rand(rows, cols)
matrix_list = matrix.tolist()

def center_lists():
    # среднее по строкам через list comprehension
    means = [sum(row)/len(row) for row in matrix_list]
    result = []
    for i, row in enumerate(matrix_list):
        m = means[i]
        result.append([x - m for x in row])
    return result

def center_numpy():
    # broadcasting: вычитаем среднее по строке, сохраняя размерность
    means = matrix.mean(axis=1, keepdims=True)
    return matrix - means

t_center_list = measure_time(center_lists)
t_center_np = measure_time(center_numpy)
print(f"списки:   {t_center_list:.3f} сек")
print(f"numpy:    {t_center_np:.3f} сек")
print(f"ускорение: {t_center_list/t_center_np:.1f}x")

# ------------------------------------------------------------
# 5. использование памяти
# ------------------------------------------------------------

print("\n" + "="*60)
print("5. сравнение потребления памяти")
print("="*60)

size_mem = 10_000_000
list_int = list(range(size_mem))
np_int32 = np.arange(size_mem, dtype=np.int32)
np_int64 = np.arange(size_mem, dtype=np.int64)
np_float64 = np.arange(size_mem, dtype=np.float64)

# getsizeof показывает только overhead списка, а не содержимого.
# для списка размер элементов считается отдельно.
def total_size_of_list(lst):
    size = sys.getsizeof(lst)
    for item in lst:
        size += sys.getsizeof(item)
    return size

print(f"list of ints (python objects): {total_size_of_list(list_int) / 1024 / 1024:.1f} MB")
print(f"numpy int32:                   {np_int32.nbytes / 1024 / 1024:.1f} MB")
print(f"numpy int64:                   {np_int64.nbytes / 1024 / 1024:.1f} MB")
print(f"numpy float64:                 {np_float64.nbytes / 1024 / 1024:.1f} MB")

# ------------------------------------------------------------
# 6. вычисление тригонометрических функций
# ------------------------------------------------------------

print("\n" + "="*60)
print("6. вычисление синуса для 10 млн чисел")
print("="*60)

x_list = [i * 0.001 for i in range(10_000_000)]
x_arr = np.arange(0, 10000, 0.001, dtype=np.float32)

def sin_list():
    return [math.sin(v) for v in x_list]

def sin_numpy():
    return np.sin(x_arr)

t_sin_list = measure_time(sin_list)
t_sin_np = measure_time(sin_numpy)
print(f"списки + math.sin: {t_sin_list:.3f} сек")
print(f"numpy.sin:         {t_sin_np:.3f} сек")
print(f"ускорение:         {t_sin_list/t_sin_np:.1f}x")

# ------------------------------------------------------------
# 7. scipy.linalg: решение СЛАУ (сравнение с методом Гаусса)
# ------------------------------------------------------------

print("\n" + "="*60)
print("7. решение системы линейных уравнений (500x500)")
print("="*60)

n_solve = 500
# генерируем случайную матрицу и правую часть
np.random.seed(42)
A_solve = np.random.rand(n_solve, n_solve)
b_solve = np.random.rand(n_solve)

# решение через scipy.linalg.solve (использует LAPACK)
def solve_scipy():
    return linalg.solve(A_solve, b_solve)

# наивный метод Гаусса на чистом python (ужасно медленно)
def gauss_elimination(A, b):
    n = len(b)
    # копируем, чтобы не портить исходные
    A = [row[:] for row in A]
    b = b[:]
    for i in range(n):
        # поиск максимума для устойчивости
        max_row = i
        for k in range(i+1, n):
            if abs(A[k][i]) > abs(A[max_row][i]):
                max_row = k
        A[i], A[max_row] = A[max_row], A[i]
        b[i], b[max_row] = b[max_row], b[i]
        # нормировка
        pivot = A[i][i]
        for j in range(i, n):
            A[i][j] /= pivot
        b[i] /= pivot
        # вычитание из нижних строк
        for k in range(i+1, n):
            factor = A[k][i]
            for j in range(i, n):
                A[k][j] -= factor * A[i][j]
            b[k] -= factor * b[i]
    # обратный ход
    x = [0.0]*n
    for i in range(n-1, -1, -1):
        s = b[i]
        for j in range(i+1, n):
            s -= A[i][j] * x[j]
        x[i] = s
    return x

# переводим матрицу в список списков для наивного метода
A_list_solve = A_solve.tolist()
b_list_solve = b_solve.tolist()

print("решаем через scipy.linalg.solve...")
t_scipy = measure_time(solve_scipy, repeats=5)
x_scipy = solve_scipy()

print("решаем методом Гаусса на python (может быть долго)...")
t_gauss = measure_time(gauss_elimination, A_list_solve, b_list_solve, repeats=1)
x_gauss = gauss_elimination(A_list_solve, b_list_solve)

print(f"scipy.linalg.solve: {t_scipy:.4f} сек")
print(f"python gauss:       {t_gauss:.2f} сек")
print(f"ускорение:          {t_gauss/t_scipy:.0f}x")
# небольшая проверка точности (первые 5 компонент)
print("первые 5 решений (scipy):", x_scipy[:5])
print("первые 5 решений (gauss):", x_gauss[:5])

# ------------------------------------------------------------
# 8. scipy.optimize: минимизация функции (vs ручной перебор)
# ------------------------------------------------------------

print("\n" + "="*60)
print("8. поиск минимума функции")
print("="*60)

# тестовая функция: f(x) = (x-3)^2 + sin(6*x)
def func(x):
    return (x-3)**2 + np.sin(6*x)

# ручной метод: грубый перебор с мелким шагом
def brute_force_search(x_min, x_max, step):
    best_x = x_min
    best_f = func(best_x)
    x = x_min
    while x <= x_max:
        f_val = func(x)
        if f_val < best_f:
            best_f = f_val
            best_x = x
        x += step
    return best_x, best_f

# scipy оптимизация (метод Брента)
def optimize_scipy():
    res = optimize.minimize_scalar(func, bounds=(0, 6), method='bounded')
    return res.x, res.fun

# замеры
step = 0.0001  # очень маленький шаг для точности, но медленно
print("перебор на python...")
t_brute = measure_time(brute_force_search, 0, 6, step)
x_brute, f_brute = brute_force_search(0, 6, step)

print("scipy.optimize...")
t_scipy_opt = measure_time(optimize_scipy)
x_scipy_opt, f_scipy_opt = optimize_scipy()

print(f"перебор (шаг {step}): время {t_brute:.3f} сек, минимум в x={x_brute:.5f}, f={f_brute:.5f}")
print(f"scipy.optimize:        время {t_scipy_opt:.4f} сек, минимум в x={x_scipy_opt:.5f}, f={f_scipy_opt:.5f}")
print(f"ускорение:             {t_brute/t_scipy_opt:.0f}x")

# ------------------------------------------------------------
# итоговая табличка (если есть pandas)
# ------------------------------------------------------------

if use_pandas:
    data = {
        'тест': ['сложение векторов','скалярное произведение','умножение матриц','центрирование','sin()','решение СЛАУ','оптимизация'],
        'python (списки/циклы)': [t_list, t_dot_list, t_mat_list, t_center_list, t_sin_list, t_gauss, t_brute],
        'numpy/scipy': [t_arr, t_dot_arr, t_mat_arr, t_center_np, t_sin_np, t_scipy, t_scipy_opt],
    }
    df = pd.DataFrame(data)
    df['ускорение, x'] = df['python (списки/циклы)'] / df['numpy/scipy']
    print("\n" + "="*60)
    print("сводная таблица результатов")
    print("="*60)
    print(df.round(3).to_string(index=False))

print("\nглавный вывод: numpy/scipy выигрывают за счёт:")
print("- векторизации (операции над целыми массивами на C)")
print("- непрерывного хранения в памяти (cache locality)")
print("- оптимизированных библиотек (BLAS/LAPACK, FFTW и др.)")
print("- готовых высокоэффективных алгоритмов (минимация, интегрирование, решение СЛАУ)")

1. поэлементное сложение двух векторов
списки:      0.294 сек
numpy:        0.006 сек
ускорение:    49.3x

2. скалярное произведение векторов
списки:      0.116 сек
numpy:        0.001 сек
ускорение:    144.1x

3. умножение матриц (1000x1000)
измеряем время (минуту терпения)...
списки (ikj): 27.64 сек
numpy (BLAS):  0.01 сек
ускорение:     5079.5x

4. центрирование матрицы (вычитание среднего по каждой строке)
списки:   0.013 сек
numpy:    0.000 сек
ускорение: 26.5x

5. сравнение потребления памяти
list of ints (python objects): 343.3 MB
numpy int32:                   38.1 MB
numpy int64:                   76.3 MB
numpy float64:                 76.3 MB

6. вычисление синуса для 10 млн чисел
списки + math.sin: 0.274 сек
numpy.sin:         0.013 сек
ускорение:         20.6x

7. решение системы линейных уравнений (500x500)
решаем через scipy.linalg.solve...
решаем методом Гаусса на python (может быть долго)...
scipy.linalg.solve: 0.0019 сек
python gauss:       1.30 сек
ускорение:         